# Generating a Peaked Random Quantum Circuit (PRQC)

This notebook generates one PRQC the same way the circuits in `circuits/` were made (paper, Sec. III).

1. **Random half R**: `rl` layers of Haar-random two-qubit unitaries in a 1D brick-wall.
2. **Peaking half P(θ)**: another `pl` brick-wall layers, also random at the start.
3. **Optimisation**: maximise $|\langle 0^n | P(\theta)^\dagger R | 0^n \rangle|^2$, i.e. the overlap of $R|0\rangle$ and $P(\theta)|0\rangle$, with the circuit treated as a tensor network in quimb.

The circuit $P(\theta)^\dagger R$ is then peaked on the all-zero bitstring. It is transpiled to `u` + `cz` and exported as OpenQASM 3.

In [1]:
from pathlib import Path

import jax
import numpy as np
import quimb as qu
import quimb.tensor as qtn
from quimb.tensor.optimize import TNOptimizer
from qiskit import QuantumCircuit, qasm3, transpile
from qiskit.quantum_info import Statevector

jax.config.update("jax_enable_x64", True)  # optimise in double precision

## Parameters

For depth `d`, the random half has `rl = d // 2` layers and the peaking half has `pl = d - rl` layers, so for odd `d` the peaking half gets the extra layer.
The paper set was generated with 5,000 L-BFGS-B iterations followed by up to 10,000 Adam iterations. The target peak probability was 0.999.

In [2]:
n = 6        # qubits
d = 10       # total depth (layers)
seed = 0

rl, pl = d // 2, d - d // 2
loss_target = -0.999
stages = [("L-BFGS-B", 5_000), ("adam", 10_000)]

out_dir = Path("generated")

## Brick-wall circuits as tensor networks

Each two-qubit gate is kept as its own tensor, tagged `G{k}`, on top of the $|0^n\rangle$ MPS. That way the optimiser can vary the gates while the input state (tag `MPS`) stays fixed.
With two qubits a brick-wall is impossible, so every layer acts on the single pair.

In [3]:
def brickwall_pairs(n, layers):
    """Qubit pairs of every gate, layer by layer."""
    pairs = []
    for r in range(layers):
        start = 0 if (r % 2 == 0 or n == 2) else 1
        pairs += [(i, i + 1) for i in range(start, n - 1, 2)]
    return pairs


def random_brickwall(n, layers):
    """|psi> = U|0^n> with U a brick-wall of Haar-random two-qubit gates."""
    psi = qtn.MPS_computational_state("0" * n)
    for i in range(n):
        psi[i].modify(left_inds=[f"k{i}"], tags=[f"I{i}", "MPS"])
    for k, pair in enumerate(brickwall_pairs(n, layers)):
        psi.gate_(qu.rand_uni(4), pair, tags={"U", f"G{k}"})
    return psi.astype_("complex128")


qu.seed_rand(seed)
rqc = random_brickwall(n, rl)   # random half R|0>
pqc = random_brickwall(n, pl)   # peaking half P(theta)|0>, to be optimised

## Optimise the peaking half

In [4]:
def negative_overlap(psi, target):
    return -abs((target.H & psi) ^ all) ** 2


def norm_fn(psi):
    # project the gate tensors back onto unitaries
    return psi.isometrize(method="mgs", allow_no_left_inds=True)


pqc_opt = pqc
for optimizer, iterations in stages:
    tnopt = TNOptimizer(
        pqc_opt,
        loss_fn=negative_overlap,
        norm_fn=norm_fn,
        loss_constants={"target": rqc},
        constant_tags=["MPS"],
        loss_target=loss_target,
        autodiff_backend="jax",
        optimizer=optimizer,
        progbar=False,
    )
    pqc_opt = tnopt.optimize(iterations, ftol=1e-16)
    peak = -float(negative_overlap(pqc_opt, rqc))
    print(f"{optimizer}: peak probability = {peak:.4f}")
    if peak >= -loss_target:
        break

L-BFGS-B: peak probability = 0.9956


adam: peak probability = 0.9956


## Convert to a Qiskit circuit

When a gate tensor `G{k}` is created it has the index order `(out_i, out_j, in_i, in_j)`, so `data.reshape(4, 4)` is the gate matrix with qubit `i` as the most significant bit.
The optimiser keeps the index names but can permute them, so every optimised gate is first transposed back into the index order of the unoptimised circuit `ref`.
Qiskit uses the opposite qubit order, so each gate is applied to `[j, i]`.

In [5]:
def gates(psi, ref, n, layers):
    for k, (i, j) in enumerate(brickwall_pairs(n, layers)):
        g = psi[f"G{k}"].transpose(*ref[f"G{k}"].inds).data.reshape(4, 4)
        u, _, vh = np.linalg.svd(g)  # nearest unitary (removes float noise)
        yield u @ vh, (i, j)


def to_qiskit(psi, ref, n, layers):
    qc = QuantumCircuit(n)
    for g, (i, j) in gates(psi, ref, n, layers):
        qc.unitary(g, [j, i])
    return qc


prqc = to_qiskit(rqc, rqc, n, rl).compose(to_qiskit(pqc_opt, pqc, n, pl).inverse())  # P(theta)^dagger R

probs = Statevector(prqc).probabilities()
print(f"P(0...0) = {probs[0]:.4f}   (optimiser: {peak:.4f})")
print(f"most likely bitstring: {np.argmax(probs):0{n}b}")

P(0...0) = 0.9956   (optimiser: 0.9956)
most likely bitstring: 000000


## Transpile and export

In [6]:
prqc_ucz = transpile(prqc, basis_gates=["u", "cz"], optimization_level=3)
print(dict(prqc_ucz.count_ops()))

path = out_dir / f"q{n}" / f"prqc_q{n}_d{d}.qasm"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(qasm3.dumps(prqc_ucz))
print("wrote", path)

{'u': 144, 'cz': 69}
wrote generated/q6/prqc_q6_d10.qasm


## Checking a published circuit

The QASM files in `circuits/` contain no measurements. On an ideal device each one is peaked on `0…0`, with the probability listed in `circuits/metrics.csv`.
Loading QASM 3 requires `qiskit-qasm3-import`.

In [7]:
qc = qasm3.load("circuits/q6/prqc_q6_d10.qasm")
print(f"P(0...0) = {Statevector(qc).probabilities()[0]:.4f}")

P(0...0) = 0.9989
